# PDF -> Markdown bằng Chunkr (lumina-ai-inc/chunkr)

Pipeline thay thế cho `to_MD.ipynb` (Marker). Chunkr làm **layout analysis + OCR + semantic chunking** ở phía server.

## ⚠️ CẢNH BÁO PRIVACY (đọc trước khi chạy)
- Cell dưới dùng **Chunkr CLOUD** (`chunkr.ai`): file PDF được **gửi lên server bên thứ ba**.
- `02_sources/` chứa **full-text sách bản quyền** — đây là lý do repo phải để **private**.
- Gửi sách lên cloud có thể vi phạm bản quyền/ToS. Cân nhắc:
  - Chỉ dùng cloud cho tài liệu **public** (paper open-access, báo cáo IMF/BIS công khai).
  - Với sách bản quyền: **self-host Chunkr bằng Docker** (giữ dữ liệu local) — xem mục cuối notebook.

## Lấy API key
Đăng ký free tier tại https://chunkr.ai -> Dashboard -> API Keys. Free tier có **giới hạn số trang/tháng** — sách ~240 trang có thể vượt quota.

In [ ]:
!pip install -q chunkr-ai
import chunkr_ai
print("chunkr-ai:", getattr(chunkr_ai, "__version__", "installed"))

In [ ]:
# Mount Drive + thư mục input/output (tách riêng với pipeline Marker)
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

INPUT_DIR = Path('/content/gdrive/MyDrive/chunkr_input')
OUT_DIR   = Path('/content/gdrive/MyDrive/chunkr_output')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input :", INPUT_DIR)
print("Output:", OUT_DIR)

In [ ]:
# API key (KHÔNG hardcode) + khởi tạo client
import os
from getpass import getpass
from chunkr_ai import Chunkr

api_key = os.environ.get("CHUNKR_API_KEY") or getpass("Nhập CHUNKR_API_KEY: ")
# Self-host: bỏ comment 2 dòng dưới, trỏ về instance local của bạn
# os.environ["CHUNKR_URL"] = "http://localhost:8000"
# chunkr = Chunkr(api_key=api_key, url=os.environ["CHUNKR_URL"])
chunkr = Chunkr(api_key=api_key)
print("Chunkr client sẵn sàng.")

In [ ]:
# Upload PDF (giữ Ctrl/Cmd để chọn nhiều), lưu vào INPUT_DIR
from google.colab import files
print("Chọn PDF cần parse:")
uploaded = files.upload()
for name, data in uploaded.items():
    if name.lower().endswith('.pdf'):
        (INPUT_DIR / name).write_bytes(data)
        print(f"  Saved: {name}")

pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print(f"
{len(pdfs)} PDF sẵn sàng:")
for p in pdfs:
    print(f"  {p.name[:55]}  ({p.stat().st_size//1024} KB)")

In [ ]:
# (Tuỳ chọn) cấu hình chất lượng cao cho tài liệu scan/nhiều bảng + công thức
config = None
try:
    from chunkr_ai.models import Configuration, OcrStrategy
    config = Configuration(
        ocr_strategy=OcrStrategy.ALL,   # OCR mọi trang (sách scan cũ)
        high_resolution=True,           # render nét hơn -> bảng/công thức tốt hơn
    )
    print("Configuration: OCR=ALL, high_resolution=True")
except Exception as e:
    print(f"Bỏ cấu hình tuỳ chọn, dùng mặc định ({type(e).__name__}: {e})")

In [ ]:
# Parse từng PDF -> Markdown. upload() ở SDK sync sẽ CHỜ task xong rồi trả về.
import time

failed = []
for src in sorted(INPUT_DIR.glob('*.pdf')):
    out_path = OUT_DIR / f"{src.stem}.md"
    print(f"
=== {src.name} ===")
    t0 = time.time()
    try:
        task = chunkr.upload(str(src), config)   # config=None -> mặc định
        print(f"  task_id={getattr(task,'task_id','?')} status={getattr(task,'status','?')}")

        # Xuất markdown gộp toàn bộ segment -> ghi thẳng ra file
        task.markdown(output_file=str(out_path))

        size = out_path.stat().st_size // 1024 if out_path.exists() else 0
        print(f"  OK -> {out_path.name} ({size} KB) trong {time.time()-t0:.0f}s")
    except Exception as e:
        print(f"  FAIL: {type(e).__name__} - {e}")
        failed.append(src.name)

if failed:
    print(f"
⚠ FAIL: {failed}  (kiểm tra quota/định dạng/page-limit)")
else:
    print("
Tất cả OK.")

In [ ]:
# Tải kết quả .md về máy
from google.colab import files
for md_file in sorted(OUT_DIR.glob('*.md')):
    print(f"  {md_file.name} ({md_file.stat().st_size//1024} KB)")
    files.download(str(md_file))

In [ ]:
# Dọn dẹp + đóng client
import shutil
chunkr.close()
# Xoá input đã xử lý (giữ output trên Drive)
# shutil.rmtree(INPUT_DIR, ignore_errors=True); INPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Đã đóng Chunkr client.")

## So với pipeline Marker (`to_MD.ipynb`)
| | Marker (to_MD) | Chunkr (file này) |
|---|---|---|
| Compute | GPU Colab (tải model ~GB) | Cloud server (không cần GPU) |
| Privacy | **local**, dữ liệu không rời máy | **gửi lên cloud** (rủi ro bản quyền) |
| Output | Markdown + LaTeX `$$` | Markdown + HTML + **chunks sẵn** (hợp KG2RAG) |
| Chi phí | miễn phí (Colab) | free tier giới hạn trang, trả phí khi vượt |
| Bảng/công thức | texify tốt | layout analysis + VLM |

## Self-host (giữ dữ liệu local — cho sách bản quyền)
```bash
git clone https://github.com/lumina-ai-inc/chunkr
cd chunkr
# cấu hình LLM models trong .env, rồi:
docker compose up -d
# API: http://localhost:8000 | UI: http://localhost:5173
```
Sau đó ở cell init: set `CHUNKR_URL=http://localhost:8000` và truyền vào `Chunkr(api_key=..., url=...)`.

## Sau khi có .md
Chạy hậu xử lý OCR như pipeline cũ:
```
python 05_scripts/normalize_ocr_md.py "<file>.md" --scan
python 05_scripts/normalize_ocr_md.py "<file>.md" --apply
```